# Module 04 — Data Visualization: Module Assessment (Solution)

**Save Your Work**

Before you begin, save a copy of this notebook to your Google Drive:
**File > Save a copy in Drive**

---

This is the **solution notebook**. It contains complete, working code for all tasks
along with expected outputs shown as comments.

## Setup — Create the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

np.random.seed(42)
n = 200
continents = ['Africa', 'Americas', 'Asia', 'Europe', 'Oceania']
cities = [f"City_{i}" for i in range(1, n+1)]
years = list(range(2018, 2024))

records = []
for city_id in range(1, 41):
    continent = continents[city_id % 5]
    base_gdp = np.random.uniform(10000, 80000)
    base_qol = np.random.uniform(40, 90)
    for year in years:
        records.append({
            'city': f"City_{city_id}",
            'continent': continent,
            'year': year,
            'population': int(np.random.uniform(500000, 10000000)),
            'gdp_per_capita': round(base_gdp * np.random.uniform(0.95, 1.1), 0),
            'avg_temperature': round(np.random.uniform(5, 35), 1),
            'quality_of_life': round(base_qol + (year - 2018) * np.random.uniform(-0.5, 1.5), 1),
        })

df = pd.DataFrame(records)
df_recent = df[df['year'] == df['year'].max()].copy()
print(df.shape)
# Expected output: (240, 7)
df.head()

---

## Task 1: Line Chart — Quality of Life Over Time (Matplotlib)

In [ ]:
# Task 1: Line chart — avg quality of life per continent over years

qol_by_year = df.groupby(['year', 'continent'])['quality_of_life'].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 6))

for continent in continents:
    subset = qol_by_year[qol_by_year['continent'] == continent]
    ax.plot(subset['year'], subset['quality_of_life'], marker='o', label=continent)

ax.set_title('Average Quality of Life by Continent (2018\u20132023)')
ax.set_xlabel('Year')
ax.set_ylabel('Avg Quality of Life')
ax.legend(title='Continent')
ax.grid(True)
plt.tight_layout()
plt.show()

**Trend observation:** Quality of life scores across all continents remain relatively
stable from 2018 to 2023, with modest upward or flat trends depending on the continent,
reflecting the small per-year random increments built into the dataset.

---

## Task 2: Horizontal Bar Chart — GDP per Capita by Continent (Matplotlib)

In [ ]:
# Task 2: Horizontal bar chart — avg GDP per capita by continent, sorted high to low

gdp_by_continent = (
    df_recent
    .groupby('continent')['gdp_per_capita']
    .mean()
    .sort_values(ascending=True)  # ascending=True puts highest at top for barh
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(gdp_by_continent.index, gdp_by_continent.values, color='steelblue')
ax.set_title('Average GDP per Capita by Continent (Most Recent Year)')
ax.set_xlabel('Avg GDP per Capita (USD)')
ax.set_ylabel('Continent')
plt.tight_layout()
plt.show()

---

## Task 3: Scatter Plot — GDP vs Quality of Life (Matplotlib)

In [ ]:
# Task 3: Scatter plot — GDP per capita vs quality of life, colored by continent

colors = {'Africa': 'tab:blue', 'Americas': 'tab:orange', 'Asia': 'tab:green',
          'Europe': 'tab:red', 'Oceania': 'tab:purple'}

fig, ax = plt.subplots(figsize=(9, 6))

for continent, group in df_recent.groupby('continent'):
    ax.scatter(
        group['gdp_per_capita'],
        group['quality_of_life'],
        label=continent,
        color=colors[continent],
        alpha=0.8
    )

ax.set_title('GDP per Capita vs Quality of Life')
ax.set_xlabel('GDP per Capita (USD)')
ax.set_ylabel('Quality of Life')
ax.legend(title='Continent')
plt.tight_layout()
plt.show()

**Relationship observation:** There is no strong linear relationship between GDP per capita
and quality of life in this synthetic dataset, as cities across all GDP levels show a wide
range of quality-of-life scores driven largely by the continent's random base value.

---

## Task 4: Subplots — Population and Temperature Distributions (Matplotlib)

In [ ]:
# Task 4: Subplots — histogram of population (left) and avg_temperature (right)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: population histogram
axes[0].hist(df_recent['population'], bins=15, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of City Population')
axes[0].set_xlabel('Population')
axes[0].set_ylabel('Frequency')

# Right: avg_temperature histogram
axes[1].hist(df_recent['avg_temperature'], bins=15, color='coral', edgecolor='white')
axes[1].set_title('Distribution of Average Temperature')
axes[1].set_xlabel('Avg Temperature (\u00b0C)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

---

## Task 5: Interactive Scatter Plot (Plotly Express)

In [ ]:
# Task 5: Interactive scatter plot using Plotly Express

fig = px.scatter(
    df_recent,
    x='gdp_per_capita',
    y='quality_of_life',
    color='continent',
    hover_data=['city'],
    template='plotly_white',
    title='GDP per Capita vs Quality of Life (Interactive)',
    labels={
        'gdp_per_capita': 'GDP per Capita (USD)',
        'quality_of_life': 'Quality of Life',
        'continent': 'Continent'
    }
)
fig.show()

---

## Task 6: Interactive Grouped Bar Chart (Plotly Graph Objects)

In [ ]:
# Task 6: Interactive grouped bar chart using Plotly Graph Objects

summary = df_recent.groupby('continent').agg(
    avg_gdp=('gdp_per_capita', 'mean'),
    avg_qol=('quality_of_life', 'mean')
).reset_index()

fig = go.Figure()

fig.add_trace(go.Bar(
    x=summary['continent'],
    y=summary['avg_gdp'],
    name='Avg GDP per Capita (USD)'
))

fig.add_trace(go.Bar(
    x=summary['continent'],
    y=summary['avg_qol'],
    name='Avg Quality of Life'
))

fig.update_layout(
    barmode='group',
    title='Avg GDP per Capita and Quality of Life by Continent',
    xaxis_title='Continent',
    yaxis_title='Average Value'
)

fig.show()

---

## Task 7: Summary of Insights

**Insights:**

- **Line chart (Task 1):** Quality of life trends are largely flat across all continents
  from 2018 to 2023, with no continent showing a dramatic sustained increase or decrease,
  suggesting stable conditions in this dataset.
- **Horizontal bar chart (Task 2):** There is meaningful variation in average GDP per
  capita across continents in the most recent year, with the highest-GDP continent earning
  roughly two to three times the average of the lowest-GDP continent.
- **Interactive scatter plot (Task 5):** Even within the same continent, individual cities
  span a wide range of quality-of-life scores regardless of their GDP level, highlighting
  that economic output alone does not predict wellbeing in this dataset.